In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DB_PATH, FIGURES_DIR, ensure_dirs

ensure_dirs()

from src.viz import apply_chinese, get_font_prop

font_prop = get_font_prop()
plt.rcParams["axes.unicode_minus"] = False

con = duckdb.connect(str(DB_PATH), read_only=True)


In [ ]:
path = con.sql("""
WITH user_flags AS(
    SELECT 
        user_id,
        MAX(CASE WHEN behavior_type = '浏览' THEN 1 ELSE 0 END) AS has_pv,
        MAX(CASE WHEN behavior_type = '收藏' THEN 1 ELSE 0 END) AS has_fav,
        MAX(CASE WHEN behavior_type = '加购' THEN 1 ELSE 0 END) AS has_cart,
        MAX(CASE WHEN behavior_type = '购买' THEN 1 ELSE 0 END) AS has_buy
    FROM user_behavior
    GROUP BY user_id
)

SELECT 
    has_pv || has_fav || has_cart || has_buy AS path,
    COUNT(*) AS user_cnt
FROM user_flags
GROUP BY path
ORDER BY user_cnt DESC
""").df()
path

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
sns.barplot(data=path, x = "path", y = "user_cnt", ax=ax, color="#55a868")
for i,v in enumerate(path["user_cnt"]):
    ax.text(i,v, f"{v:,}", ha="center", va = "bottom", fontsize=9)
ax.set_title("用户行为组合分布")
ax.set_xlabel("路径（浏览/收藏/加购/购买)")
ax.tick_params(axis="x", rotation=45)
apply_chinese(ax, font_prop)
plt.savefig(FIGURES_DIR/"behavior_path.png")
plt.show()